 # Protostellar Outflows DATA Pipeline

This project aims to retrieve ALMA observations of protostellar outflows from the ALMA Science Archive using TAP queries and build a sample of sources exhibiting episodic outflow morphology. Our initial goal is to identify observations of the SiO (8–7) molecular line, which traces shocked gas associated with high-velocity protostellar jets and outflows. Once we have established a sample of sources with SiO (8–7) observations, we will expand our search to include the CO (3–2) and CO (2–1) transitions to further investigate the morphology of protostellar outflows.

# These commands are helpful to see astroquery version and relevant steuptools


In [51]:
#import pkg_resources
#print(pkg_resources.__file__)
#pip install setuptools
#import sys
#print(sys.executable)
#print(sys.version)

# Import Libraries

In [74]:
from astroquery.alma import Alma
import os
import numpy as np
import pandas as pd
from astroquery.alma.utils import parse_frequency_support
import astropy.units as u
from astropy.table import Table
import shutil
from pathlib import Path
import tarfile
from astropy.io import fits
from spectral_cube import SpectralCube
import matplotlib.pyplot as plt 
#import alminer
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import add_beam
from glob import glob
from urllib.parse import urlparse, unquote
from collections import defaultdict
import re

import glob

# Use ALMA AstroTAP Queries to Identify Data Products Based on Science Keywords in the ALMA Science Archive

In [59]:
# Edit this part for the specific science keyword
query = """
SELECT DISTINCT
    target_name,
    member_ous_uid,
    band_list,
    frequency_support
FROM ivoa.obscore
WHERE
    LOWER(science_keyword) LIKE '%outflows, jets and ionized winds%' 
    AND band_list = '7'
    AND antenna_arrays LIKE '%:D%:D%'
    AND antenna_arrays NOT LIKE '%:CM%'
    AND science_observation = 'T'
    AND UPPER(scan_intent) LIKE '%TARGET%'
"""

results = alma.query_tap(query).to_table()

# Print unique source names
source_names = sorted(set(results["target_name"]))

for source in source_names:
    print(source)

print("\nTotal unique source names:", len(source_names))

08303-4303
10295-5746
10365-5803
AFGL5142_H2O
B1c
BHR71
CW_Tau
Cha-MMS1
DG_Tau
G014.33-0.64_H2O
G015.03-0.67_H2O
G034.43+0.24MM1_H2O
G035.02+0.34_H2O
G206.93-16.61W2
G209.55-19.68S1
G210.37-19.53S
G328.2551-0.5321_A
G331.512-0.103
G34.3+0.2
G358.93
G45
G45.07+0.13
G45.12+0.13
G5.89
HH1177
HH211
HH212
HH_111
HH_211
HH_30
HH_900
I14382
I14498
I15520
I15596
I16060
I16071
I16076
I16272
I16351
I17204
I17220
IC348
IRAS-4C
IRAS05168+3634_H2O
IRAS06058+2138_H2O
IRAS06061+2151_H2O
IRAS18537+0749_H2O
IRAS_16272-4837
IRAS_16293-2422
IRAS_16547-4247
IRAS_18089-1732
ISO-Oph_102
K3-50A
L1448-mm
Lh01
Lh02
Lh03
Lh04
Lh05
Lh06
Lh07
Lh08
Lh09
Lh10
Ll01
Ll02
Ll03
Ll04
Ll05
Ll06
Ll07
Ll08
Ll09
Ll10
Lm01
Lm02
Lm03
Lm04
Lm05
Lm06
Lm07
Lm08
Lm09
Lm10
M17-UC1
MMS5OMC-3
MMS_1
MWC349A
MonR2-IRS2
NGC1333-IRAS2A
NGC6334I
NGC6334IN_H2O
NGC6334I_H2O
Northern_Arm
Orion-KL
Orion_KL_Field_1_Orion_Hot_Core
Orion_KL_Field_2_SMA1
Orion_KL_Field_3_North-west_Clump
PN_M3-28
RAS05553+1631_H2O
S01
S02
S03
S04
S05
S06
S07
S08

# Now edit observed frequency for any molecule and find related obseravtions

In [60]:
obs_freq = 347.33  # Observed frequency in GHz (SiO 8-7)

matching_rows = []

for i, row in enumerate(results):

    ranges = parse_frequency_support( #parse frequency support)
        row["frequency_support"]
    )

    for low, high in ranges:

        if low.to_value(u.GHz) <= obs_freq <= high.to_value(u.GHz):
            matching_rows.append(i)
            break

sio_obs = results[matching_rows]

# Print every matching row, including repeated source names
source_names = sorted(sio_obs["target_name"])

for source in source_names:
    print(source)

print("\nTotal matching rows:", len(sio_obs))
print("Total unique source names:", len(set(source_names)))

08303-4303
10295-5746
10365-5803
B1c
B1c
G206.93-16.61W2
G209.55-19.68S1
G210.37-19.53S
G328.2551-0.5321_A
G328.2551-0.5321_A
G331.512-0.103
HH211
HH211
HH212
HH212
HH212
HH212
HH212
HH212
HH212
HH_111
HH_211
IC348
IC348
IRAS-4C
IRAS-4C
IRAS_16293-2422
IRAS_16293-2422
IRAS_18089-1732
L1448-mm
L1448-mm
Lh01
Lh02
Lh03
Lh04
Lh05
Lh06
Lh07
Lh08
Lh09
Lh10
Ll01
Ll02
Ll03
Ll04
Ll05
Ll06
Ll07
Ll08
Ll09
Ll10
Lm01
Lm02
Lm03
Lm04
Lm05
Lm06
Lm07
Lm08
Lm09
Lm10
MMS5OMC-3
NGC1333-IRAS2A
NGC1333-IRAS2A
Orion_KL_Field_1_Orion_Hot_Core
Orion_KL_Field_2_SMA1
Orion_KL_Field_3_North-west_Clump
S01
S02
S03
S04
S05
S06
S07
S08
S09
S10
W42-MME

Total matching rows: 78
Total unique source names: 64


# Some of the LMC/SMC observations found with this query but we can exclude those observations here:

In [61]:

# Names to exclude: Lh01, Ll01, Lm01, S01, SC01, etc.
exclude = r"(?:Lh|Ll|Lm|S|SC)\d+"

keep_rows = [
    not bool(re.fullmatch(
        exclude,
        str(name).strip(),
        flags=re.IGNORECASE
    ))
    for name in sio_obs["target_name"]
]

selected_obs = sio_obs[keep_rows]

# Print remaining source names
source_names = sorted(set(selected_obs["target_name"]))

for source in source_names:
    print(source)

print("\nTotal selected source names:", len(source_names))

# Use these IDs for the later product/download steps
sio_obs_ids = sorted(set(selected_obs["member_ous_uid"]))

08303-4303
10295-5746
10365-5803
B1c
G206.93-16.61W2
G209.55-19.68S1
G210.37-19.53S
G328.2551-0.5321_A
G331.512-0.103
HH211
HH212
HH_111
HH_211
IC348
IRAS-4C
IRAS_16293-2422
IRAS_18089-1732
L1448-mm
MMS5OMC-3
NGC1333-IRAS2A
Orion_KL_Field_1_Orion_Hot_Core
Orion_KL_Field_2_SMA1
Orion_KL_Field_3_North-west_Clump
W42-MME

Total selected source names: 24


# These sources may contain SiO observations according to the observed frequency

In [63]:
# Unique observation IDs 
sio_obs_ids = sorted(set(
    selected_obs["member_ous_uid"]
))

# Expand products
products = alma.get_data_info(
    sio_obs_ids,
    expand_tarfiles=True
)

print("Available product rows:", len(products))
print(products.colnames)

Available product rows: 3471
['ID', 'access_url', 'service_def', 'error_message', 'semantics', 'local_semantics', 'description', 'content_type', 'content_length', 'content_qualifier', 'link_auth', 'link_authorized']


In [64]:
for url in products["access_url"][:10]:
    print(url)

https://almascience.nrao.edu/dataPortal/member.uid___A001_X122_X4a1.README.txt
https://almascience.nrao.edu/dataPortal/2013.1.00018.S_uid___A001_X122_X4a1_001_of_001.tar
https://almascience.nrao.edu/dataPortal/2013.1.00018.S_uid___A001_X122_X4a1_auxiliary.tar
https://almascience.nrao.edu/dataPortal/2013.1.00018.S_uid___A002_Xa916fc_Xfad.asdm.sdm.tar
https://almascience.nrao.edu/dataPortal/2013.1.00018.S_uid___A001_X122_X4a1_external_ari_l_001_of_001.tar
https://almascience.nrao.edu/dataPortal/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_1_2_3_342243MHz.12m.cont.I.mask.fits.gz
https://almascience.nrao.edu/dataPortal/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_1_2_3_342243MHz.12m.cont.I.pb.fits.gz
https://almascience.nrao.edu/dataPortal/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_1_2_3_342243MHz.12m.cont.I.pbcor.fits
https://almascience.nrao.edu/dataPortal/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347323MHz.12m.mfs.I.mask.fits.

# Download the data for all files and keep only PB, PBCOR and Mask fits files

In [65]:

selected_files = []
seen_urls = set()

for row in products:

    url = str(row["access_url"])
    filename = unquote(Path(urlparse(url).path).name)
    name = filename.lower()

    # Keep science products
    if "_sci." not in name:
        continue

    # Keep FITS cubes
    if ".cube." not in name:
        continue

    if not name.endswith((".fits", ".fits.gz")):
        continue

    # Select related product types
    if ".cube.i.pbcor." in name:
        file_type = "PBCOR"

    elif ".cube.i.pb." in name:
        file_type = "PB"

    elif ".cube.i.mask." in name:
        file_type = "MASK"

    else:
        continue

    # Skip duplicate URLs
    if url in seen_urls:
        continue

    seen_urls.add(url)

    selected_files.append({
        "filename": filename,
        "url": url,
        "type": file_type
    })

print("Total selected FITS products:", len(selected_files))

for file_type in ["PB", "PBCOR", "MASK"]:
    print(
        f"{file_type}:",
        sum(f["type"] == file_type for f in selected_files)
    )

Total selected FITS products: 369
PB: 129
PBCOR: 129
MASK: 111


In [66]:
for file in selected_files:
    print(file["type"], "|", file["filename"])

MASK | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.mask.fits.gz
PB | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pb.fits.gz
PBCOR | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
PB | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pb.fits.gz
PBCOR | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits
MASK | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.mask.fits.gz
PB | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pb.fits.gz
PBCOR | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits
PB | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pb.fits.gz
PBCOR | member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pbcor.fits
MASK | member.uid___

# Prepare to download files

In [67]:
from pathlib import Path


download_dir = Path(
    "/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files"
)

download_dir.mkdir(parents=True, exist_ok=True)


alma.cache_location = str(download_dir)

print("Download directory:", download_dir)

Download directory: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files


# Download only missing files

In [69]:

download_urls = []
seen_urls = set()

for file in selected_files:

    url = file["url"]
    filename = file["filename"]

    # Skip duplicate URLs
    if url in seen_urls:
        continue

    seen_urls.add(url)

    # Skip files already downloaded
    if (download_dir / filename).exists():
        print("Already downloaded:", filename)
        continue

    download_urls.append(url)

print("\nSelected FITS products:", len(selected_files))
print("Unique download URLs:", len(seen_urls))
print("Files to download:", len(download_urls))

# Download missing files
if download_urls:

    downloaded_files = alma.download_files(
        download_urls,
        savedir=str(download_dir)
    )

    print("\nDownload step completed.")

else:
    downloaded_files = []
    print("\nAll selected files already exist.")

Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.mask.fits.gz
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pb.fits.gz
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pb.fits.gz
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.mask.fits.gz
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pb.fits.gz
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits
Already downloaded: member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pb.fits.

KeyboardInterrupt: 

# After downloading all FITS files, inspect the spectral windows of each cube to identify which ones cover the observed frequency of the target molecular line, since the downloaded data may also contain spectral windows of other molecules.

In [70]:

# Related PB, PBCOR, and MASK files share this base name
def cube_group(filename):
    return re.sub(
        r"\.(pbcor|pb|mask)\.fits(?:\.gz)?$",
        "",
        filename,
        flags=re.IGNORECASE
    )

keep_groups = set() #groups to keep in the observations

pbcor_files = sorted(
    list(download_dir.glob("*.cube.I.pbcor.fits")) +
    list(download_dir.glob("*.cube.I.pbcor.fits.gz"))
)

for file in pbcor_files:

    try:
        cube = SpectralCube.read(str(file), use_dask=True)

        # Read frequency coordinates using the cube's spectral metadata
        frequencies = cube.with_spectral_unit(
            u.GHz
        ).spectral_axis.to_value(u.GHz)

        freq_min = frequencies.min()
        freq_max = frequencies.max()

        freq_covers = (
            freq_min <= obs_freq <= freq_max
        )

        if freq_covers:
            keep_groups.add(cube_group(file.name))

        print(
            file.name,
            "|", f"{freq_min:.4f}–{freq_max:.4f} GHz",
            "|", "KEEP" if freq_covers else "Outside range"
        )

    except Exception as error:
        print("Needs review:", file.name, "|", error)

member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits | 347.2687–347.3856 GHz | KEEP
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits | 348.1575–349.8451 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits | 337.3347–337.4516 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pbcor.fits | 334.6410–336.3286 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pbcor.fits | 347.2710–347.3879 GHz | KEEP


member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw1_349004MHz.12m.cube.I.pbcor.fits | 348.1598–349.8474 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw2_337395MHz.12m.cube.I.pbcor.fits | 337.3370–337.4538 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw3_335487MHz.12m.cube.I.pbcor.fits | 334.6432–336.3309 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pbcor.fits | 347.2693–347.3861 GHz | KEEP
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw1_349002MHz.12m.cube.I.pbcor.fits | 348.1580–349.8457 GHz | Outside range


member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw2_337394MHz.12m.cube.I.pbcor.fits | 337.3352–337.4521 GHz | Outside range
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw3_335485MHz.12m.cube.I.pbcor.fits | 334.6415–336.3292 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits | 347.2688–347.3856 GHz | KEEP
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw1_349000MHz.12m.cube.I.pbcor.fits | 348.1565–349.8438 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits | 337.3347–337.4516 GHz | Outside range


member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw3_335486MHz.12m.cube.I.pbcor.fits | 334.6424–336.3297 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pbcor.fits | 347.2698–347.3867 GHz | KEEP
member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw1_349001MHz.12m.cube.I.pbcor.fits | 348.1575–349.8449 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw2_337394MHz.12m.cube.I.pbcor.fits | 337.3358–337.4526 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw3_335487MHz.12m.cube.I.pbcor.fits | 334.6434–336.3307 GHz | Outside range


member.uid___A001_X122_X4a3.ari_l.J1924-2914_bp.spw0_347321MHz.12m.cube.I.pbcor.fits | 347.2626–347.3794 GHz | KEEP
member.uid___A001_X122_X4a3.ari_l.J1924-2914_bp.spw1_348994MHz.12m.cube.I.pbcor.fits | 348.1503–349.8376 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.J1924-2914_bp.spw2_337387MHz.12m.cube.I.pbcor.fits | 337.3288–337.4456 GHz | Outside range
member.uid___A001_X122_X4a3.ari_l.J1924-2914_bp.spw3_335480MHz.12m.cube.I.pbcor.fits | 334.6364–336.3237 GHz | Outside range


member.uid___A001_X133d_X3486._G328.2551-0.5321_A__sci.spw27.cube.I.pbcor.fits | 345.3862–347.2575 GHz | Outside range
member.uid___A001_X133d_X3488._G328.2551-0.5321_A__sci.spw25.cube.I.pbcor.fits | 347.2173–349.0908 GHz | KEEP
member.uid___A001_X133d_X3488._G328.2551-0.5321_A__sci.spw27.cube.I.pbcor.fits | 345.3844–347.2579 GHz | Outside range
member.uid___A001_X133d_X3488._G328.2551-0.5321_A__sci.spw29.cube.I.pbcor.fits | 335.3430–337.2165 GHz | Outside range
member.uid___A001_X133d_X3488._G328.2551-0.5321_A__sci.spw31.cube.I.pbcor.fits | 333.4681–335.3416 GHz | Outside range


... (36 outputs left out)

member.uid___A001_X2d20_Xec0._G210.37-19.53S__sci.spw33.cube.I.pbcor.fits | 357.5530–359.4233 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw0_347332MHz.12m.cube.I.pbcor.fits | 347.3026–347.3611 GHz | KEEP


member.uid___A001_X88f_X259.ari_l.HH212_sci.spw10_336671MHz.12m.cube.I.pbcor.fits | 336.6416–336.7000 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw11_337398MHz.12m.cube.I.pbcor.fits | 337.3684–337.4268 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw12_335022MHz.12m.cube.I.pbcor.fits | 334.0862–335.9588 GHz | Outside range


member.uid___A001_X88f_X259.ari_l.HH212_sci.spw1_345797MHz.12m.cube.I.pbcor.fits | 345.7681–345.8266 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw2_346204MHz.12m.cube.I.pbcor.fits | 346.1749–346.2333 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw3_345921MHz.12m.cube.I.pbcor.fits | 345.8913–345.9497 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw4_347000MHz.12m.cube.I.pbcor.fits | 346.9704–347.0288 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw5_346653MHz.12m.cube.I.pbcor.fits | 346.6242–346.6826 GHz | Outside range


member.uid___A001_X88f_X259.ari_l.HH212_sci.spw6_345905MHz.12m.cube.I.pbcor.fits | 345.8761–345.9345 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw7_346552MHz.12m.cube.I.pbcor.fits | 346.5226–346.5810 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw8_337062MHz.12m.cube.I.pbcor.fits | 337.0331–337.0915 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.HH212_sci.spw9_336555MHz.12m.cube.I.pbcor.fits | 336.5258–336.5843 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.J0541-0211_chk.spw0_347332MHz.12m.cube.I.pbcor.fits | 347.3032–347.3616 GHz | KEEP
member.uid___A001_X88f_X259.ari_l.J0541-0211_chk.spw4_347000MHz.12m.cube.I.pbcor.fits | 346.9710–347.0294 GHz | Outside range
member.uid___A001_X88f_X259.ari_l.J0552p0313_ph.spw0_347330MHz.12m.cube.I.pbcor.fits | 347.3003–347.3588 GHz | KEEP


# Once the SiO spectral windows are identified, save the corresponding FITS files for generating moment 0 maps later

In [71]:
keep_files = []

for file in sorted(download_dir.iterdir()):

    # Only these three FITS product types
    if not re.search(
        r"\.cube\.i\.(pbcor|pb|mask)\.fits(?:\.gz)?$",
        file.name,
        flags=re.IGNORECASE
    ):
        continue

    if cube_group(file.name) in keep_groups:
        keep_files.append(file)

print("\nRelated files to keep:\n")

for file in keep_files:
    print(file.name)

print("\nMatching cube groups:", len(keep_groups))
print("Related FITS files:", len(keep_files))


Related files to keep:

member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.mask.fits.gz
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pb.fits.gz
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pb.fits.gz
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pb.fits.gz
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.mask.fits.gz
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pb.fits.gz
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pb.fits.gz


# Check all files in the scratch folder

In [75]:
# Check all files in the scratch_directory
all_files = glob.glob("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/*.fits*")
#all_files
#print(Path(all_files[0]).name)

# Keep files with SiO lines

In [76]:
keep_files_names = [Path(f).name for f in keep_files]

print("keep_files:")
for f in keep_files:
    print(repr(f))
    #print(len(f))


print("\nall_files:")
for f in all_files:
    print(repr(f))

keep_files:
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.mask.fits.gz')
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pb.fits.gz')
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits')
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pb.fits.gz')
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pbcor.fits')
PosixPath('/scratch/gsc9zr/Machine-Learning-Project/Astromorph_

# Move SiO lines files in the final fits folder

In [77]:
for y in all_files:
    if Path(y).name in keep_files_names :
        destination = Path(y.replace('scratch_files', 'final_fits_file'))
        #print(destination)
        # Make sure destination directory exists
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.move(Path(y), destination)
        print("Moved:", y)

Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a3.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X15a1_X1a12.NGC1333-IRAS2A_sci.spw39.cube.I.mask.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X15a1_X1a14.B1c_sci.spw39.cube.I.mask.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X88f_X259.ari_l.J0541-0211_chk.spw0_347332MHz.12

# Now delete all other files from the scratch folder

In [ ]:
remaining_files = glob.glob('./alma_TAP/scratch_files/*')

for y in remaining_files:
    #os.remove(y)
    print("Deleted:", y)
print(len(remaining_files))